# EDA cho Silver

Notebook nay chi doc cac bang Silver da cau hinh trong repo va hien thi:

- duong dan table
- schema
- vai dong dau cua dataframe

Muc tieu la xem du lieu trong tung bang co gi, khong lam them phan tich phu.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
        REPO_ROOT = candidate
        break

for path in (REPO_ROOT, REPO_ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from apps.spark.common import load_profile
from learnlake.runtime import build_spark

spark = build_spark("SilverEDA")
profile = load_profile("daotao_ai")
spark


In [ ]:
silver_tables = {"silver_event_index": profile.silver.event_index.path}
for target in profile.silver.targets.values():
    silver_tables[target.table] = target.path
if profile.silver.invalid is not None:
    silver_tables[profile.silver.invalid.table] = profile.silver.invalid.path

print("Silver tables to inspect:")
for name, path in sorted(silver_tables.items()):
    print(f"- {name}: {path}")


In [ ]:
def inspect_delta_table(table_name: str, path: str, sample_rows: int = 10):
    return spark.read.format("delta").load(path).select("*")


In [ ]:
tables_loaded = {}
for table_name, path in sorted(silver_tables.items()):
    try:
        tables_loaded[table_name] = inspect_delta_table(table_name, path)
    except Exception as exc:
        print(f"\n[SKIP] {table_name}: {exc}")

tables_loaded


## Cach chay

- Chay cell khoi tao Spark va `profile` truoc.
- Sau do chay cell tao `silver_tables`.
- Cuoi cung chay vong lap `tables_loaded` de xem toan bo bang.
- Neu ban chi muon xem 1 bang, goi truc tiep:

```python
inspect_delta_table("silver_event_index", profile.silver.event_index.path)
```

- Neu muon xem nhieu dong hon, tang tham so `sample_rows` trong `inspect_delta_table`.
- `spark.read.format("delta").load(path)` la phan chinh de doc dataframe tu tung table.
